In [1]:
import os
import sys
import subprocess
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from pyarrow import fs
import pyarrow.parquet as pq
import pandas as pd
import numpy as np

os.environ["HADOOP_CONF_DIR"] = "/path/to/hadoop/conf/"
os.environ["JAVA_HOME"] = "/path/to/java"
os.environ["HADOOP_HOME"] = "/path/to/hadoop"
os.environ["ARROW_LIBHDFS_DIR"] = "/path/to/hadoop/lib/"
os.environ["CLASSPATH"] = subprocess.check_output(
    "$HADOOP_HOME/bin/hadoop classpath --glob", shell=True
).decode("utf-8")

hdfs = fs.HadoopFileSystem(
    host="hdfs://your-hdfs-host", port=8020
)

# import pandas as pd
from tqdm import tqdm
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

warnings.simplefilter('ignore', ConvergenceWarning)
import sys

if not sys.warnoptions:
    import warnings

    warnings.simplefilter("ignore")

import logging

logging.getLogger("prophet").setLevel(logging.WARNING)
logging.getLogger("cmdstanpy").disabled = True

from tqdm import trange

2024-01-11 20:25:12,041 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.seasonal import STL
from prophet import Prophet
import pandas as pd
from pmdarima import auto_arima


def measure_simility(arr1, arr2):
    arr1 = arr1.reshape((-1,))
    arr2 = arr2.reshape((-1,))
    score = np.corrcoef(arr1, arr2)[0, 1]
    return score


def pick_data_train(vt, arrs, n):
    score = []
    for arr in arrs:
        score.append(measure_simility(vt, arr))
    score = np.array(score)
    return np.argsort(score)[-n:], score[np.argsort(score)[-n:]]


def create_data_total(df, month, year, lag, shift):
    x_knowns = df[df.index < year].iloc[:, 11 + month - lag - shift: 11 + month - shift].values.reshape((-1, lag))
    # y_known = df[df.index == year].iloc[0,11 + month-lag: 11 +month].values
    y_known = df[df.index == year].iloc[0, 11 + month - lag: 11 + month].values
    # y_known = df.iloc[-1, 11 + month - lag: 11 + month].values
    x_news = df[df.index < year].iloc[:, 11 + month - shift].values
    # print(month,year,y_known)
    return x_knowns, y_known, x_news


def create_data_cumsum(df, month, year, lag, shift):
    x_knowns = df[df.index < year].iloc[:, 11 + month - lag - shift: 11 + month - shift].cumsum(axis=1).values.reshape(
        (-1, lag))
    y_known = df[df.index == year].iloc[:, 11 + month - lag: 11 + month].cumsum(axis=1).iloc[0, :].values
    # y_known = df.iloc[[-1], 11 + month - lag: 11 + month].cumsum(axis=1).iloc[0, :].values
    x_news = df[df.index < year].iloc[:, 11 + month - lag - shift: 12 + month - shift].cumsum(axis=1).iloc[:, -1].values
    return x_knowns, y_known, x_news


def create_data_rolling(df, month, year, lag, shift, n_rolling):
    y_old = df[df.index == year].iloc[0, 11 + month - 1 - n_rolling + 1:11 + month - 1].sum()
    df = df.rolling(n_rolling, axis=1).mean()
    x_knowns = df[df.index < year].iloc[:, 11 + month - lag - shift: 11 + month - shift].values.reshape((-1, lag))
    y_known = df[df.index == year].iloc[0, 11 + month - lag: 11 + month].values
    x_news = df[df.index < year].iloc[:, 11 + month - shift].values
    return x_knowns, y_known, x_news, y_old


def predict_total(known_y, known_x, new_x, order):
    # known_y = known_y.reshape((-1,))
    # known_x = known_x.reshape((-1,))
    coefficients = np.polyfit(known_x, known_y, order)
    estimated_func = np.poly1d(coefficients)
    return float(estimated_func(new_x))


def predict_cumsum(known_y, known_x, new_x, order):
    # known_y = known_y.reshape((-1,))
    # known_x = known_x.reshape((-1,))
    coefficients = np.polyfit(known_x, known_y, order)
    estimated_func = np.poly1d(coefficients)
    return float(estimated_func(new_x))


def predict(df_total, year, month: int, lag: int, order1: int, order2: int, shift: int, n_max):
    x_knowns, y_known, x_news = create_data_total(df_total, month, year, lag, shift)
    max_scores, coeffs = pick_data_train(y_known, x_knowns, n_max)
    pre = []
    for max_score in max_scores:
        x_knowns, y_known, x_news = create_data_total(df_total, month, year, lag, shift)
        x_known = x_knowns[max_score]
        x_new = x_news[max_score]

        pred1 = predict_total(known_y=y_known,
                              known_x=x_known,
                              new_x=x_new, order=order1)
        pre.append(pred1)
        shift = 0
        x_knowns, y_known, x_news = create_data_cumsum(df_total, month, year, lag, shift)
        x_known = x_knowns[max_score]
        x_new = x_news[max_score]
        pred2 = predict_cumsum(known_y=y_known,
                               known_x=x_known,
                               new_x=x_new, order=order2) - y_known[-1]

        pre.append(pred2)
    # if (month==1)&(lag==12):
    #     return np.mean(pre),coeffs.mean()
    # else:
    # n_rolling = 2
    # x_knowns,y_known,x_news,y_old = create_data_rolling(df_total,month,year,lag,shift,n_rolling)
    # max_scores,coeffs = pick_data_train(y_known,x_knowns,n_max)
    # for max_score in max_scores:
    #     x_known = x_knowns[max_score]
    #     x_new = x_news[max_score]
    #
    #     pred3 = predict_total(known_y=y_known,
    #                           known_x=x_known,
    #                           new_x=x_new,order=order1)*n_rolling - y_old
    #     pre.append(pred3)
    return np.mean(pre), coeffs.mean()


def predict_master(data: pd.DataFrame, d_date: pd.DataFrame, month, year, lag, p, d, q, n_max, type_decomp,
                   methods_season, method_resid, method_trend):
    date = f"{year}-{month}-1"
    # print(date)
    # d_train = data[data.index < f"{year}-1-1"]
    d_train = data[data.index < date]
    d_train = pd.concat([d_train, d_date], axis=1).dropna(axis=0)

    d_residuals, d_trend, d_seasonal = detrend(d_train, type=type_decomp)
    # y_residual=0
    y_residual = forecast_residual(d_residuals.values, p, d, q, method_resid=method_resid)
    # except:
    #     y_residual = 0

    d_seasonal.name = name_province
    d_seasonal = pd.concat([d_seasonal, d_date], axis=1).fillna(0)
    d_seasonal = d_seasonal.pivot_table(columns='month', index='year', values=name_province)
    y_seasonal = forecast_seasonal(d_seasonal, month, year, lag, n_max, methods_season)

    d_trend.name = name_province
    d_trend = pd.concat([d_trend, d_date], axis=1).dropna()
    try:
        y_trend = forecast_trend(d_trend.iloc[-2, 0], d_trend.iloc[-1, 0], method_trend=method_trend)
    except:
        y_trend = 0

    # d_residuals = pd.concat([d_residuals, d_date], axis=1).dropna()
    # y_re3 = d_seasonal.pivot_table(columns='month', index='year', values=name_province).dropna(axis=0).mean(axis=0)[
    #     month]
    # d_residuals = d_residuals.pivot_table(columns='month', index='year', values=name_province)
    # d_trend = d_trend.pivot_table(columns='month', index='year', values=name_province)
    # y_re2 = forecast_seasonal(d_trend, month, year, lag, 2)
    # return y_lr + (y_re1 +y_re2+y_re)/3,y_re
    # return (y_re3 + y_re1) / 2 + y_re2
    return y_trend, y_seasonal, y_residual


def detrend(df: pd.DataFrame, type='STL'):
    if type == 'STL':
        data = df.copy()
        data = data.iloc[:, 0]
        stl = STL(data,period=12)
        res = stl.fit()
        trend = res.trend
        resid = res.resid
        season = res.seasonal
        return resid, trend, season
    if type == 'prophet':
        data = df.copy()
        data = data.iloc[:, [0]]
        data = data.reset_index()
        data.columns = ['ds', 'y']
        model = Prophet(yearly_seasonality=12)
        model.fit(data)
        forecaster = model.predict(model.make_future_dataframe(periods=0, freq='MS'))
        forecaster['y'] = data['y']
        forecaster['resid'] = forecaster.apply(lambda row: (row.y - (row.yearly + row.trend)), axis=1)
        forecaster = forecaster.set_index('ds')
        df_resid = forecaster['resid']
        df_trend = forecaster['trend']
        df_season = forecaster['yearly']
        df_season.name = 'seasonal'
        return df_resid, df_trend, df_season
    else:
        return None, None, None


def compute_error(data, month, year, y_pr):
    y_tr = data[data.index == f'{year}-{month}-1'].values[0]
    return (y_pr - y_tr) / y_tr, y_tr


def forecast_seasonal(data, month, year, lag, n_max, method='statistics'):
    if method == 'statistics':
        df = data.copy()
        df = pd.concat([df.shift(1, axis=0), df, df.shift(-1, axis=0)], axis=1).iloc[1:, :] + 10e10
        # lag = int(params[month - 1][1])
        # shift = int(params[month - 1][2])
        # order1 = int(params[month - 1][3])
        # order2 = int(params[month - 1][4])
        # lag = 6
        shift = 0
        order1 = 1
        order2 = 1
        y, _ = predict(df, year, month, lag, order1, order2, shift, n_max)
        return y - 10e10
    if method == 'average':
        df = data.copy()
        df = df[df.index<year]
        # print(df.dropna(axis=0).mean(axis=0))
        y = df.dropna(axis=0).mean(axis=0)[month]
        # print(y)
        return y
    if method == 'latest':
        df = data.copy()
        df = df[df.index<year]
        # print(df.dropna(axis=0).mean(axis=0))
        y = df.dropna(axis=0).iloc[-1, :][month]
        # print(y)
        return y
    else:
        raise "Not specify the season method"


def forecast_trend(y_1, y_t, method_trend='average'):
    if method_trend == 'average':
        y_pr_trend = y_t + (y_t - y_1)
        # y_pr_trend = y_t
        return y_pr_trend
    if method_trend == 'latest':
        return y_t
    else:
        raise "Not specify the trend method"


def forecast_residual(series, p, d, q, method_resid='arima'):
    if method_resid == 'arima':
        try:
            model = ARIMA(series, order=(p, d, q))
            model_fit = model.fit()
            return model_fit.forecast(freq='MS')[0]
        except:return 0
    # if method_resid == 'autoarima':
    #     model = auto_arima(series, seasonal=False, trace=False)
    #     return model.predict(n_periods=1, freq='MS')[0]
    else:
        return 0


def get_ytr(df, month, year):
    try:
        filter = f'{year}-{month}-1'
        return df[df.index == filter].values[0][0]
    except:
        return None


In [9]:
pd.read_parquet('/path/to/company_data/sa/cpc/store/data/SUM_TOTAL/total_sl_13_provinces_2023-12.parquet',filesystem=hdfs).sort_values('PHIEN',ascending=False).set_index('PHIEN').iloc[:,:-1]

,BDINH,DANANG,DLAC,DNONG,GLAI,KHOA,KTUM,PYEN,QBINH,QNAM,QNGAI,QTRI,TTHUE
PHIEN,,,,,,,,,,,,,
2023-12-01,209589687,251941410,192772258,68487295,129279584,220393105,53503395,91446014,76917535,198279228,177979839,69479002,168910328
2023-11-01,237220530,306060760,181575737,65081029,123236212,245231088,46903748,81903276,92552312,225265298,188298784,78325276,166285061
2023-10-01,229283628,325046776,177891988,55271575,104950394,268993300,45397486,107902396,89358961,216527628,191251580,69936919,174538640
2023-09-01,245846739,334854205,176154962,54350765,110230293,260440322,43585903,100944943,99008566,236540705,205012048,77915611,178232208
2023-08-01,242371738,343783477,163510885,56349981,102942041,265621102,41849276,102249857,110559224,240644513,215322023,76426513,199147336
2023-07-01,233239947,326930522,157939171,53774306,99639058,257318544,35575725,98089406,113635119,235583294,208659512,76274477,198860925
2023-06-01,228831703,330679786,168895712,53361487,107430058,271527719,36919148,96108003,110711603,227748124,195321222,76746089,195194018
2023-05-01,216810167,286065814,187296839,60365365,117478037,247775311,38476870,92239501,94808312,218040740,200881892,66616139,175082939
2023-04-01,201071762,263577819,199440776,65812059,130184998,227387246,41138185,86218995,87052560,207999642,171806536,63538594,171897527


In [47]:
date_range = pd.date_range(start='2014-1-1', end='2025-12-1', freq='MS')
df_datetime = pd.DataFrame({'stt': [i for i in range(date_range.shape[0])]}, index=date_range)
df_datetime['month'] = df_datetime.index.month
df_datetime['year'] = df_datetime.index.year
df_datetime

,stt,month,year
2014-01-01,0,1,2014
2014-02-01,1,2,2014
2014-03-01,2,3,2014
2014-04-01,3,4,2014
2014-05-01,4,5,2014
...,...,...,...
2025-08-01,139,8,2025
2025-09-01,140,9,2025
2025-10-01,141,10,2025
2025-11-01,142,11,2025


In [4]:
dfp = pd.read_parquet('/path/to/company_data/user/<user>/cpc/data/parameter/final_params_13province_u2022.parquet',filesystem=hdfs)
dfp

,type_decomp,method_season,method_resid,method_trend,province,n_max,error_abs
0,STL,statistics,None,average,BDINH,5,0.014079
1,STL,statistics,arima,average,DANANG,5,0.053772
2,STL,statistics,None,average,DLAC,5,0.066493
3,STL,statistics,None,latest,DNONG,5,0.072136
4,STL,statistics,arima,latest,GLAI,5,0.045875
5,STL,statistics,arima,latest,KHOA,5,0.047504
6,STL,statistics,None,average,KTUM,2,0.031834
7,STL,statistics,None,average,PYEN,2,0.033658
8,STL,statistics,None,latest,QBINH,2,0.032650
9,STL,statistics,None,average,QNAM,5,0.026136


In [51]:
path_to_folder_hdfs = (
    "/path/to/company_data/user/<user>/cpc/data/u2023/detail/DVDC3/" )

folder = [
    f.path for f in hdfs.get_file_info(fs.FileSelector(path_to_folder_hdfs)) ]
folder

['/path/to/company_data/user/<user>/cpc/data/u2023/detail/DVDC3/QBINH.parquet',
 '/path/to/company_data/user/<user>/cpc/data/u2023/detail/DVDC3/KTUM.parquet',
 '/path/to/company_data/user/<user>/cpc/data/u2023/detail/DVDC3/DLAC.parquet',
 '/path/to/company_data/user/<user>/cpc/data/u2023/detail/DVDC3/GLAI.parquet',
 '/path/to/company_data/user/<user>/cpc/data/u2023/detail/DVDC3/QTRI.parquet',
 '/path/to/company_data/user/<user>/cpc/data/u2023/detail/DVDC3/BDINH.parquet',
 '/path/to/company_data/user/<user>/cpc/data/u2023/detail/DVDC3/TTHUE.parquet',
 '/path/to/company_data/user/<user>/cpc/data/u2023/detail/DVDC3/QNAM.parquet',
 '/path/to/company_data/user/<user>/cpc/data/u2023/detail/DVDC3/DANANG.parquet',
 '/path/to/company_data/user/<user>/cpc/data/u2023/detail/DVDC3/QNGAI.parquet',
 '/path/to/company_data/user/<user>/cpc/data/u2023/detail/DVDC3/DNONG.parquet',
 '/path/to/company_data/user/<user>/cpc/data/u2023/detail/DVDC3/KHOA.parquet',
 '/path/to/company_data/user/<user>/cpc/data/

In [16]:
dict_df_total = {}
folder = sorted(folder)
for file in tqdm(folder):
    df = pd.read_parquet(file,filesystem=hdfs)
    dict_df_total[file] = df

100%|██████████| 13/13 [00:00<00:00, 18.18it/s]


In [29]:
df_result = pd.DataFrame()
years = [2023]
for file in tqdm(folder):
    df_total = dict_df_total[file].copy()
    # df_total = df_total.fillna(0)
    # df_total['Month'] = df_total.index.month
    # df_total['Year'] = df_total.index.year
    # df_total_padding = pd.DataFrame({col :None for col in df_total.columns},index = pd.date_range(start='2023-9-1',end='2026-1-1',freq='MS'))
    # df_total_final = pd.concat([df_total,df_total_padding],axis=0)
    df_total_final = df_total
    dp = dfp[dfp.province ==file.split('/')[-1].split('.')[0]]
    lag = 6
    n_max = dp.n_max.values[0]
    type_decomp = dp.type_decomp.values[0]
    method_season =dp.method_season.values[0]
    method_trend =dp.method_trend.values[0]
    method_resid =dp.method_resid.values[0]
    # method_resid = 'None'
    p = 1
    d = 0
    q = 1
    lst_pr = df_total_final.columns
    for n_province in range(len(lst_pr)):
        name_province = lst_pr[n_province]
        for year in years:
            for month in range(8,11):
                if (month==10)&(year==2023):
                    break
                # try:
                # print(month)
                lst_ypr = []
                lst_ytr = []
                lst_trend = []
                lst_resid = []
                lst_season = []
                df = df_total_final.copy().iloc[:, [n_province]]
                df = df.sort_index()
                for t in range(1):
                    # try:
                    # print(t)
                    if (int(month + t) > 12):
                        year = year+1
                        month = month - 12
                        y_trend, y_seasonal, y_residual = predict_master(
                            data=df,
                            d_date=df_datetime,
                            month=month+t, year=year,
                            lag=lag, p=p, d=d, q=q,
                            n_max=n_max,
                            type_decomp=type_decomp,
                            methods_season=method_season, method_resid=method_resid,
                            method_trend=method_trend)
                        y_tr = get_ytr(df, month + t, year)
                        y_pr = y_trend + y_seasonal + y_residual

                        df.at[f"{year}-{month+t}-1", name_province] = y_pr
                        year = year - 1
                        month = month + 12
                    else:
                        y_trend, y_seasonal, y_residual = predict_master(
                            data=df,
                            d_date=df_datetime,
                            month=month+t, year=year,
                            lag=lag, p=p, d=d, q=q,
                            n_max=n_max,
                            type_decomp=type_decomp,
                            methods_season=method_season, method_resid=method_resid,
                            method_trend=method_trend)

                        y_tr = get_ytr(df, month + t, year)
                        y_pr = y_trend + y_seasonal + y_residual

                        df.at[f"{year}-{month+t}-01", name_province] = y_pr

                    lst_ypr.append(y_pr)
                    lst_ytr.append(y_tr)
                    lst_trend.append(y_trend)
                    lst_season.append(y_seasonal)
                    lst_resid.append(y_residual)

                # except:
                #
                #     print(year,month,t)
                #     break
                df_result = pd.concat([df_result, pd.DataFrame({
                    'file':[file],
                    'province': [name_province],
                    'year':[year],
                    'month': [month],
                    'trend':[lst_trend],
                    'season':[lst_season],
                    'resid':[lst_resid],
                    'y_pr': [lst_ypr],
                    'y_tr': [lst_ytr],
                    'n_max':[n_max],
                })])
            # except:break


100%|██████████| 13/13 [4:18:49<00:00, 1194.61s/it] 


In [30]:
def expand_ypr(row):
    row = row['y_pr']
    return row[0], row[1], row[2],row[3],row[4],row[5],row[6],row[7],row[8],row[9],row[10],row[11]
def expand_ytr(row):
    row = row['y_tr']
    return row[0], row[1], row[2],row[3],row[4],row[5],row[6],row[7],row[8],row[9],row[10],row[11]

df_result[['y_pr1', 'y_pr2', 'y_pr3','y_pr4','y_pr5','y_pr6','y_pr7','y_pr8','y_pr9','y_pr10','y_pr11','y_pr12',]] = df_result.apply(expand_ypr, axis=1, result_type='expand')
df_result[['y_tr1', 'y_tr2', 'y_tr3','y_tr4','y_tr5','y_tr6','y_tr7','y_tr8','y_tr9','y_tr10','y_tr11','y_tr12',]] = df_result.apply(expand_ytr, axis=1, result_type='expand')

df_result['P'] = df_result.file.apply(lambda x:x.split('/')[-1].split('.')[0])

In [44]:
df_result

,file,province,year,month,trend,season,resid,y_pr,y_tr,n_max,...,y_tr4,y_tr5,y_tr6,y_tr7,y_tr8,y_tr9,y_tr10,y_tr11,y_tr12,P
0,/path/to/company_data/user/...,NONE - NONE- NONE,2022,1,"[18710699.41054707, 18023268.6310926, 17075760...","[-4248254.463912964, -7710018.540252686, -4026...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[14462444.946634106, 10313250.090839915, 13049...","[16195370.0, 15519642.0, 16472498.0, 17980180....",5,...,17980180.0,18896938.0,20863068.0,20962329.0,20669191.0,20323735.0,18961988.0,16718401.0,16018662.0,BDINH
0,/path/to/company_data/user/...,NONE - NONE- NONE,2022,2,"[18366391.236336563, 17492126.569598768, 16237...","[-7612356.548065186, -4091390.7444000244, -952...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[10754034.688271377, 13400735.825198743, 15284...","[15519642.0, 16472498.0, 17980180.0, 18896938....",5,...,18896938.0,20863068.0,20962329.0,20669191.0,20323735.0,18961988.0,16718401.0,16018662.0,12997410.0,BDINH
0,/path/to/company_data/user/...,NONE - NONE- NONE,2022,3,"[18435726.520147502, 17280834.463350736, 16224...","[-4334203.175628662, -1423337.6967468262, 3696...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[14101523.34451884, 15857496.76660391, 1659419...","[16472498.0, 17980180.0, 18896938.0, 20863068....",5,...,20863068.0,20962329.0,20669191.0,20323735.0,18961988.0,16718401.0,16018662.0,12997410.0,15151544.0,BDINH
0,/path/to/company_data/user/...,NONE - NONE- NONE,2022,4,"[17750292.428598642, 16720825.991845135, 15838...","[-1660128.2228393555, 430535.65629577637, 3749...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[16090164.205759287, 17151361.64814091, 195885...","[17980180.0, 18896938.0, 20863068.0, 20962329....",5,...,20962329.0,20669191.0,20323735.0,18961988.0,16718401.0,16018662.0,12997410.0,15151544.0,14717177.0,BDINH
0,/path/to/company_data/user/...,NONE - NONE- NONE,2022,5,"[17095053.16213133, 16267480.732278965, 155296...","[407931.9066925049, 4015777.0573425293, 420554...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[17502985.068823837, 20283257.789621495, 19735...","[18896938.0, 20863068.0, 20962329.0, 20669191....",5,...,20669191.0,20323735.0,18961988.0,16718401.0,16018662.0,12997410.0,15151544.0,14717177.0,17736564.0,BDINH
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
0,/path/to/company_data/user/...,Tỉnh Thừa Thiên Huế - Thị Xã Hương Trà- Xã Hươ...,2023,5,"[826875.3167019408, 835282.5440124782, 841026....","[19879.72543334961, 152215.96328735352, 123482...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[846755.0421352905, 987498.5072998317, 964509....","[864205.0, 1000709.0, 988168.0, 1002941.0, nan...",2,...,1002941.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TTHUE
0,/path/to/company_data/user/...,Tỉnh Thừa Thiên Huế - Thị Xã Hương Trà- Xã Hươ...,2023,6,"[838737.6733664336, 845117.4125655096, 846641....","[152683.21047973633, 123802.80200195312, 14488...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[991420.8838461699, 968920.2145674627, 991529....","[1000709.0, 988168.0, 1002941.0, nan, nan, nan...",2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TTHUE
0,/path/to/company_data/user/...,Tỉnh Thừa Thiên Huế - Thị Xã Hương Trà- Xã Hươ...,2023,7,"[846956.4795741219, 848768.101909445, 849683.7...","[123797.23989868164, 145275.4677886963, 79988....","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[970753.7194728035, 994043.5696981413, 929672....","[988168.0, 1002941.0, nan, nan, nan, nan, nan,...",2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TTHUE
0,/path/to/company_data/user/...,Tỉnh Thừa Thiên Huế - Thị Xã Hương Trà- Xã Hươ...,2023,8,"[852216.1667861855, 853545.0392151263, 854949....","[144626.7897491455, 80043.06889343262, -117375...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]","[996842.956535331, 933588.1081085589, 737573.7...","[1002941.0, nan, nan, nan, nan, nan, nan, nan,...",2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,TTHUE


In [42]:
# df_result.to_parquet('/path/to/company_data/user/<user>/cpc/result/202309/fct_12t_dvdc3_13province_012022_082023.parquet',filesystem=hdfs,index=False)

In [39]:
df_result.groupby(['file','year','month']).sum().iloc[:,7:].to_excel('1.xlsx')

In [45]:
pd.read_parquet('/path/to/company_data/user/<user>/cpc/result/202309/fct_12t_sum_13province_012022_082023.parquet',filesystem=hdfs)

,province,year,month,trend,season,resid,y_pr,y_tr,n_max,y_pr1,...,y_tr3,y_tr4,y_tr5,y_tr6,y_tr7,y_tr8,y_tr9,y_tr10,y_tr11,y_tr12
0,BDINH,2022,1,"[190367504.5659067, 190743225.32530418, 191579...","[-23557849.14945984, -35468102.77290344, -2082...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[166809655.41644686, 155275122.55240074, 17075...","[164006809.0, 162960987.0, 173845510.0, 190087...",5,1.668097e+08,...,173845510.0,190087114.0,194939087.0,214933520.0,214340300.0,210101997.0,211674343.0,196093463.0,185111618.0,180297098.0
1,BDINH,2022,2,"[190188256.1633996, 190940767.13772696, 188720...","[-35449558.038238525, -20875430.352005005, -45...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[154738698.12516108, 170065336.78572196, 18420...","[162960987.0, 173845510.0, 190087114.0, 194939...",5,1.547387e+08,...,190087114.0,194939087.0,214933520.0,214340300.0,210101997.0,211674343.0,196093463.0,185111618.0,180297098.0,145891669.0
2,BDINH,2022,3,"[192568797.0756114, 190491625.1692396, 1885090...","[-21442918.454071045, -5113138.112808228, 2804...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[171125878.62154037, 185378487.05643138, 19131...","[173845510.0, 190087114.0, 194939087.0, 214933...",5,1.711259e+08,...,194939087.0,214933520.0,214340300.0,210101997.0,211674343.0,196093463.0,185111618.0,180297098.0,145891669.0,170132715.0
3,BDINH,2022,4,"[191030117.89846572, 189075186.02137238, 18808...","[-5400686.055618286, 2777729.66217041, 2315158...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[185629431.84284744, 191852915.6835428, 211235...","[190087114.0, 194939087.0, 214933520.0, 214340...",5,1.856294e+08,...,214933520.0,214340300.0,210101997.0,211674343.0,196093463.0,185111618.0,180297098.0,145891669.0,170132715.0,167387920.0
4,BDINH,2022,5,"[189957816.6241376, 189135972.72325414, 187049...","[2934632.405670166, 23825670.468963623, 219716...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[192892449.02980778, 212961643.19221777, 20902...","[194939087.0, 214933520.0, 214340300.0, 210101...",5,1.928924e+08,...,214340300.0,210101997.0,211674343.0,196093463.0,185111618.0,180297098.0,145891669.0,170132715.0,167387920.0,193261663.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
268,TTHUE,2023,5,"[166599317.9539593, 167182924.1469203, 1675347...","[3632905.374282837, 26074166.328063965, 261835...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[170232223.32824212, 193257090.47498426, 19371...","[175060653.0, 195177149.0, 198830608.0, 199127...",2,1.702322e+08,...,198830608.0,199127317.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
269,TTHUE,2023,6,"[168138963.65267402, 168689821.7182279, 168791...","[26321024.88394165, 26340899.610229492, 249660...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[194459988.53661567, 195030721.3284574, 193757...","[195177149.0, 198830608.0, 199127317.0, nan, n...",2,1.944600e+08,...,199127317.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
270,TTHUE,2023,7,"[168831821.0350159, 168956270.5221429, 1681877...","[26342923.902893066, 25005249.24331665, 148445...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[195174744.93790898, 193961519.76545954, 18303...","[198830608.0, 199127317.0, nan, nan, nan, nan,...",2,1.951747e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
271,TTHUE,2023,8,"[169680139.2457965, 168999049.77330542, 170711...","[24872356.076965332, 17045502.200942993, -9656...","[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...","[194552495.32276183, 186044551.9742484, 161054...","[199127317.0, nan, nan, nan, nan, nan, nan, na...",2,1.945525e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
